In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
from astropy.wcs import WCS
from astropy.io import fits

In [ ]:
import scopesim as sim

This notebook shows new features of the MICADO spectroscopy implementation in ScopeSim. In particular, we look at the new spectroscopic mode `SPEC`, which uses a `SlitWheel` effect to allow changing slits easily. The `SlitWheel` (really a focal-plane mask wheel) is now also available in the imaging modes and thus allows through-slit imaging as well as the use of a pinhole-grid mask.
The previous modes `SPEC_3000x16` etc. are still available. They also use `SlitWheel` but have default settings as advertised by the mode name; scripts and notebooks using these modes should therefore continue to work as before.

If you have not done so already, please download the relevant instrument packages using the following code in a new cell:

```sim.download_packages(["Armazones", "ELT", "MICADO"])```

Alternatively, if you would like to keep the instrument packages in a separate directory, you can set the following config value:

```sim.set_inst_pkgs_path("path/to/packages")```


## Imaging of the available slits

To see all the available slits, we instantiate the `OpticalTrain` in imaging mode:

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["SCAO", "IMG_4mas"])
mcd = sim.OpticalTrain(cmd)

The effects list has the `SlitWheel` effect in the `MICADO` group; as for the `FilterWheel` effects the name of the current slit is given in parentheses. In imaging mode, the default is `False`, i.e. no slit is inserted.

In [ ]:
mcd.effects.show_in_notebook()

The slit wheel holds a number of predefined slits; these can be displayed with

In [ ]:
mcd['slit_wheel'].slits

In addition to the nominal slits (centred at 0, 0), there are offset slits shifted by 1.2 arcsec in the dispersion direction. They are meant to fill in the gaps in the spectra due to the gaps between the chips in the detector array.

To change the slit, use `mcd['slit_wheel'].change_filter()`. We'll do imaging observations with all slits, using a blank-sky background.

In [ ]:
_, axes = plt.subplots(3, 2, figsize=(7, 10))
for slit, ax in zip(mcd['slit_wheel'].slits, axes.flatten()):
    mcd['slit_wheel'].change_slit(slit)
    mcd.observe()
    ax.imshow(mcd.image_planes[0].data, origin='lower')
    ax.text(1000, 1000, slit, va='top', ha='right', c='white') 

Note that these simulations used a detector window of 1024x1024 pixels rather than the full MICADO array. The long slit extends over 15000mas/4mas = 3750 pixels in is therefore cut in these image.

## Through-slit imaging of a source
Through-slit images are taken for target acquisition but can also be used to investigate slit losses due to parts of the PSF falling outside the slits. Here we observe a point source.

In [ ]:
star = sim.source.source_templates.star(flux=15*u.mag)

In [ ]:
mcd['slit_wheel'].change_slit("3000x48")
mcd.observe(star)

In [ ]:
plt.imshow(mcd.image_planes[0].data, norm='log', origin='lower')
plt.xlim(480, 550)
plt.ylim(500, 526);

For the offset slit the source has to be shifted by 1.2 arcseconds in the y-direction.

In [ ]:
star.shift(dy=1.2)

In [ ]:
mcd['slit_wheel'].change_slit("3000x48_offset")
mcd.observe(star)

In [ ]:
plt.imshow(mcd.image_planes[0].data, norm='log', origin='lower')
plt.xlim(480, 550)
plt.ylim(800, 826);     # shift by 300 pixels (=1.2 arcsec)

## Spectroscopy with nominal and offset slits
For spectroscopy, we instantiate the `OpticalTrain` in the `SPEC` mode; the default slit is `3000x16`, the default band is `IJ`. For simplicity, we will observe blank sky here; observation of a source should be straightforward.

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["SCAO", "SPEC"])
mcd = sim.OpticalTrain(cmd)

We are going to simulate the full detector array now. Beware that spectroscopic simulations take a long time to run; as we're observing blank sky we can save a little time by switching off the PSF effects, which do no affect flat fields anyway. We set the slit explicitely and switch to the HK band.

In [ ]:
mcd['full_detector_array'].include = True
mcd['detector_window'].include = False
mcd['relay_psf'].include = False
mcd['micado_ncpas_psf'].include = False

mcd['slit_wheel'].change_slit("3000x16")
mcd['filter_wheel_1'].change_filter("Spec_HK")

In [ ]:
mcd.effects.show_in_notebook()

In [ ]:
mcd.observe()

In [ ]:
readout = mcd.readout(dit=100, ndit=100)[0]
readout.writeto("readout_nominal.fits", overwrite=True)

Rectification should be done immediately after the readout. When the slit is changed to an offset one, the spectral traces are recomputed and rectification will give a wrong result when applied to a nominal slit!

In [ ]:
with fits.open("readout_nominal.fits") as readout_nominal:
    rectified = mcd['micado_spectral_traces'].rectify_traces(readout_nominal, -1.5, 1.5)
    rectified.writeto("rect_nominal.fits", overwrite=True)

The rectified spectra that we have just created have no data at wavelengths that fall between the detectors in the detector array. To fill in these regions we perform another simulation, this time with the offset slit.

In [ ]:
mcd['slit_wheel'].change_slit("3000x16_offset")
mcd.observe()

In [ ]:
readout = mcd.readout(dit=100, ndit=100)[0]
readout.writeto("readout_offset.fits", overwrite=True)

In [ ]:
with fits.open("readout_offset.fits") as readout_offset:
    rectified = mcd['micado_spectral_traces'].rectify_traces(readout_offset, -1.5, 1.5)
    rectified.writeto("rect_offset.fits", overwrite=True)

Careful comparison of the sky lines in the detector images reveals the offset between the simulations with the nominal and the offset slit, respectively.

In [ ]:
with fits.open("readout_nominal.fits") as readout_nominal:
    with fits.open("readout_offset.fits") as readout_offset:
        _, (ax_nom, ax_off) = plt.subplots(1, 2, sharey=True)
        ax_nom.imshow(readout_nominal[5].data, origin='lower')
        ax_off.imshow(readout_offset[5].data, origin='lower')

The rectification placed the spectra on the correct linear wavelength grid. We first create the wavelength vectors using the WCS information from the FITS headers and then plot a section of the spectra.

In [ ]:
with fits.open("rect_nominal.fits") as rect_nominal:
    with fits.open("rect_offset.fits") as rect_offset:
        wcs_nom = WCS(rect_nominal[2].header).spectral
        wcs_off = WCS(rect_offset[2].header).spectral
        lam_nom = wcs_nom.all_pix2world(np.arange(rect_nominal[2].data.shape[1]), 0)[0]
        lam_off = wcs_off.all_pix2world(np.arange(rect_offset[2].data.shape[1]), 0)[0]
        spec_nom = rect_nominal[2].data[350,]
        spec_off = rect_offset[2].data[350,]

In [ ]:
plt.plot(lam_nom, spec_nom, label="nominal")
plt.plot(lam_off, spec_off+4000, label="offset (+4000)")
plt.xlim(1.6e-6, 1.63e-6)
plt.ylim(-400, 10000)
plt.legend();

The gaps in the spectra can be filled in by averaging while masking the gap regions. Here it's done using the `numpy.ma` module:

In [ ]:
mask_nom = (spec_nom == 0)
mask_off = (spec_off == 0)
mspec_nom = np.ma.masked_array(spec_nom, mask=mask_nom)
mspec_off = np.ma.masked_array(spec_off, mask=mask_off)

In [ ]:
spec_mean = np.ma.array((mspec_nom, mspec_off)).mean(axis=0)

In [ ]:
plt.plot(lam_nom, mspec_nom, label="nominal")
plt.plot(lam_off, mspec_off+3000, label="offset (+3000)")
plt.plot(lam_nom, spec_mean+6000, label="average (+6000)")
plt.xlim(1.68e-6, 1.73e-6)
plt.ylim(-400, 10400)
plt.legend();